In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader,ConcatDataset,TensorDataset
from torch.utils.data.sampler import WeightedRandomSampler
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score,classification_report
import numpy as np
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
from ekyn import *
from sklearn.model_selection import train_test_split

class ResidualBlock(nn.Module):
    def __init__(self,in_channels,out_channels,*args, **kwargs):
        super().__init__(*args, **kwargs)
        self.c1 = nn.Conv1d(in_channels=in_channels,out_channels=out_channels,kernel_size=3,padding='same')
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.c2 = nn.Conv1d(in_channels=out_channels,out_channels=out_channels,kernel_size=3,padding='same')
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.c3 = nn.Conv1d(in_channels=out_channels,out_channels=out_channels,kernel_size=3,padding='same')
        self.c4 = nn.Conv1d(in_channels=out_channels,out_channels=out_channels,kernel_size=3,padding='same')
        self.mp = nn.MaxPool1d(kernel_size=2)
    def forward(self,x):
        identity = x
        x = self.c1(x)
        x = self.bn1(x)
        x = nn.functional.relu(x)
        x = self.c2(x)
        x = self.bn2(x)
        x = nn.functional.relu(x)
        x = self.c3(x)
        x = nn.functional.relu(x)
        x = self.c4(x)
        x = x + identity
        x = nn.functional.relu(x)
        x = self.mp(x)
        return x

class SleepStageEncoder(nn.Module):
    def __init__(self, d_embedding, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.blocks = nn.Sequential(*[ResidualBlock(in_channels=1,out_channels=d_embedding)]+[ResidualBlock(in_channels=d_embedding,out_channels=d_embedding) for i in range(3)])
        self.gap = nn.AdaptiveAvgPool1d(output_size=1)
        self.classifier = nn.Linear(in_features=d_embedding,out_features=3)
    def forward(self,x):
        x = self.blocks(x)
        x = self.gap(x)
        x = x.flatten(1,2)
        # x = self.classifier(x)
        return x

def get_dataloaders(train_ids,test_ids,path_to_dataset,batch_size=32,weighted=False):
    if weighted:
        dataset = ConcatDataset([TensorDataset(*torch.load(f'{path_to_dataset}/{id}_{condition}.pt',weights_only=False)) for id in train_ids for condition in ['PF','Vehicle']])
        labels = torch.tensor([data[1].argmax().item() for data in dataset])
        class_counts = torch.bincount(labels)
        class_weights = 1. / class_counts.float()
        weights = class_weights[labels]
        trainloader = DataLoader(dataset, batch_size=batch_size, sampler=WeightedRandomSampler(weights, num_samples=len(weights), replacement=True))

        dataset = ConcatDataset([TensorDataset(*torch.load(f'{path_to_dataset}/{id}_{condition}.pt',weights_only=False)) for id in test_ids for condition in ['PF','Vehicle']])
        labels = torch.tensor([data[1].argmax().item() for data in dataset])
        class_counts = torch.bincount(labels)
        class_weights = 1. / class_counts.float()
        weights = class_weights[labels]
        testloader = DataLoader(dataset, batch_size=batch_size, sampler=WeightedRandomSampler(weights, num_samples=len(weights), replacement=True))
    else:
        trainloader = DataLoader(ConcatDataset([TensorDataset(*torch.load(f'{path_to_dataset}/{id}_{condition}.pt',weights_only=False)) for id in train_ids for condition in ['Vehicle','PF']]),batch_size=batch_size,shuffle=True)
        testloader = DataLoader(ConcatDataset([TensorDataset(*torch.load(f'{path_to_dataset}/{id}_{condition}.pt',weights_only=False)) for id in test_ids for condition in ['Vehicle','PF']]),batch_size=batch_size,shuffle=True)
    return trainloader,testloader
class SleepStageTransformer(nn.Module):
    def __init__(self,d_embedding,d_model,seq_length):
        super(SleepStageTransformer, self).__init__()
        self.d_embedding = d_embedding
        self.d_model = d_model
        self.seq_length = seq_length
        
        self.proj = nn.Linear(in_features=d_embedding,out_features=d_model)
        self.pos_embed = nn.Embedding(seq_length, d_model)
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.layer_norm1 = nn.LayerNorm(d_model)
        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),  # Typically, we increase the dimension
            nn.ReLU(),  # Non-linearity
            nn.Linear(d_model * 4, d_model)
        )
        self.fc = nn.Linear(d_model, 3)

    def forward(self, x):
        # x = x.flatten(0,1)
        # print(x.shape)
        # x = self.encoder(x)
        # print(x.shape)
        # x = x.view(-1,self.seq_length,self.d_embedding)
        # print(x.shape)
        x = self.proj(x)
        pos_embed = self.pos_embed(torch.arange(0, x.size(1)).unsqueeze(0).expand(x.size(0), -1).cuda())
        # print(pos_embed.shape)
        x = x + pos_embed
        # print(x.shape)
        q = self.query(x)
        k = self.key(x)  
        v = self.value(x)
        scores = (q @ k.transpose(-2, -1)) / (self.d_model ** 0.5)
        mask = torch.ones(self.seq_length, self.seq_length, dtype=torch.bool).cuda()
        scores = scores.masked_fill(~mask, float('-inf'))
        attention_weights = torch.softmax(scores, dim=-1)
        attention_out = attention_weights @ v
        x = self.layer_norm1(x + attention_out)
        ffn_out = self.ffn(x)
        middle_state = ffn_out[:, self.seq_length // 2, :]
        return self.fc(middle_state)
class MyDataset(torch.utils.data.Dataset):
    def __init__(self,X,y,seq_len,d_embedding):
        self.seq_len = seq_len
        self.X = torch.cat([
            torch.zeros(seq_len//2,1,d_embedding),
            X,
            torch.zeros(seq_len//2,1,d_embedding),
        ])
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx:idx+self.seq_len],self.y[idx]
path_to_experiments = f'../prototypes'
path_to_dataset = f'../pt_ekyn_robust_50hz'

train_ids,test_ids = train_test_split(ekyn_ids(),random_state=0)
print(train_ids,test_ids)

In [59]:
device = 'cuda'
d_embedding = 32
seq_length = 15
lr = 3e-4
d_model = 32
batch_size = 512

encoder = SleepStageEncoder(d_embedding=d_embedding)
criterion = nn.CrossEntropyLoss()
encoder.load_state_dict(torch.load(f'encoder.pt',weights_only=True))
for p in encoder.parameters():
    p.requires_grad = False

def get_data(id,condition):
    X,y = torch.load(f'{path_to_dataset}/{id}_{condition}.pt',weights_only=False)
    with torch.no_grad():
        encoder.to(device)
        logits = encoder(X.to(device)).cpu()
    return logits.unfold(dimension=0,size=seq_length,step=1).transpose(1,2),y[seq_length//2:-(seq_length//2)]

dataset = ConcatDataset([TensorDataset(*get_data(id,condition)) for id in train_ids for condition in ['Vehicle','PF']])
labels = torch.tensor([data[1].argmax().item() for data in dataset])
class_counts = torch.bincount(labels)
class_weights = 1. / class_counts.float()
weights = class_weights[labels]
trainloader = DataLoader(dataset, batch_size=batch_size, sampler=WeightedRandomSampler(weights, num_samples=len(weights), replacement=True))

dataset = ConcatDataset([TensorDataset(*get_data(id,condition)) for id in test_ids for condition in ['Vehicle','PF']])
labels = torch.tensor([data[1].argmax().item() for data in dataset])
class_counts = torch.bincount(labels)
class_weights = 1. / class_counts.float()
weights = class_weights[labels]
testloader = DataLoader(dataset, batch_size=batch_size, sampler=WeightedRandomSampler(weights, num_samples=len(weights), replacement=True))

# testloader = DataLoader(ConcatDataset([TensorDataset(*get_data(id,condition)) for id in test_ids for condition in ['Vehicle','PF']]),batch_size=512,shuffle=True)

model = SleepStageTransformer(d_embedding=d_embedding,d_model=d_model,seq_length=seq_length)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [ ]:
config = {
    'device':f'cuda',
    'train_ids':train_ids,
    'test_ids':test_ids,
    'trainlossi':[],
    'testlossi':[],
    'trainf1i':[],
    'testf1i':[],
    'trainf1p':[],
    'testf1p':[],
    'trainf1s':[],
    'testf1s':[],
    'trainf1w':[],
    'testf1w':[],
    'best_dev_loss':float('inf'),
    'best_dev_loss_epoch':0,
    'best_dev_f1':0,
    'best_dev_f1_epoch':0,
    'validation_frequency_epochs':1
}

model.to(config["device"])

lossi = []
best_ma_loss = float('inf')
window_size = 10
pbar = tqdm(range(2000))

for epoch in pbar:
    for Xi, yi in trainloader:
        Xi, yi = Xi.to(config["device"]), yi.to(config["device"])
        logits = model(Xi)
        loss = criterion(logits, yi)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lossi.append(loss.item())

    ma_loss = moving_average(lossi, window_size)

    if ma_loss[-1] < best_ma_loss:
        best_ma_loss = ma_loss[-1]

    pbar.set_description(f'{best_ma_loss:.3f}')

    if epoch % config["validation_frequency_epochs"] == 0:
        loss,f1,report,y_true,y_pred = evaluate(trainloader, model, criterion, config["device"])
        config["trainlossi"].append(loss)
        config["trainf1i"].append(report['macro avg']['f1-score'])
        config["trainf1p"].append(report['0']['f1-score'])
        config["trainf1s"].append(report['1']['f1-score'])
        config["trainf1w"].append(report['2']['f1-score'])
        
        loss,f1,report,y_true,y_pred = evaluate(testloader, model, criterion, config["device"])
        config["testlossi"].append(loss)
        config["testf1i"].append(report['macro avg']['f1-score'])
        config["testf1p"].append(report['0']['f1-score'])
        config["testf1s"].append(report['1']['f1-score'])
        config["testf1w"].append(report['2']['f1-score'])
        update_plot(epoch, config, loss_offset=0,  window_size=window_size)
        plt.savefig(f'loss.jpg')
        plt.close()

        if config["testlossi"][-1] < config["best_dev_loss"]:
            config["best_dev_loss"] = config["testlossi"][-1]
            config["best_dev_loss_epoch"] = epoch // config["validation_frequency_epochs"]
        if config["testf1i"][-1] > config["best_dev_f1"]:
            config["best_dev_f1"] = config["testf1i"][-1]
            config["best_dev_f1_epoch"] = epoch // config["validation_frequency_epochs"]


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay,classification_report
loss,f1,report,y_true,y_pred = evaluate(DataLoader(ConcatDataset([TensorDataset(*get_data(id,condition)) for id in train_ids for condition in ['Vehicle','PF']]),batch_size=512,shuffle=True),model,criterion,'cuda')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='true')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='pred')
print(classification_report(y_true,y_pred))

loss,f1,report,y_true,y_pred = evaluate(DataLoader(ConcatDataset([TensorDataset(*get_data(id,condition)) for id in test_ids for condition in ['Vehicle','PF']]),batch_size=512,shuffle=True),model,criterion,'cuda')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='true')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='pred')

print(classification_report(y_true,y_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay,classification_report
loss,f1,report,y_true,y_pred = evaluate(trainloader,model,criterion,'cuda')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='true')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='pred')
print(classification_report(y_true,y_pred))

loss,f1,report,y_true,y_pred = evaluate(testloader,model,criterion,'cuda')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='true')
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,normalize='pred')
print(classification_report(y_true,y_pred))